# 07 - Train a PPO Model Online

Collect `EPISODES_PER_STREAM` complete outer tasks from each FrozenLake stream, then run
`PPO_EPOCHS` × `TRAIN_STEPS` updates with the policy and value heads. Agent events store
the behavior log-probabilities. Each collection phase starts fresh environments and
clears its on-policy stores after training.

This is a short usage example, not a full experiment. The DQN inference notebook uses a
Q head; a PPO checkpoint requires reconstructing this policy head and sampling its
logits.


In [ ]:

import gymnasium as gym
import torch
from mouse_gym import EnvConfig, RepeatTaskTracker, make_group_env
from procedural_frozenlake import ensure_registered
from repeat_task_gym import RepeatTaskEnv
from torch import nn

from mouse_core import AdamW
from mouse_core.data import (
    DataLoader,
    Datastore,
    EventLog,
    Tokenizer,
    compose,
)
from mouse_core.models import (
    MLP,
    Backbone,
    DecodeCache,
    Embedding,
    Model,
    MultiEmbedding,
    action_readouts,
    prepare_decode,
    push_model_to_hub,
)
from mouse_core.objectives import (
    PpoObjective,
    boundary_discount,
    sample_discrete_action,
)

# Repositories
# Hugging Face model repo for push_model_to_hub
MODEL_ID = "mouse-example-model-ppo"

# Environment
# number of discrete actions predicted by the policy head
MAX_ACTIONS = 4
# max steps per episode
MAX_STEPS_PER_EPISODE = 24
# inner episodes per outer task
REPEAT_COUNT = 20
# number of environment streams in the GroupEnv
NUM_ENVS = 4

# Model
# HF checkpoint for Tokenizer, Embedding, and Backbone
PRETRAINED = "Qwen/Qwen3-0.6B"
HEAD_LAYERS = 1
# final policy linear's initialization
POLICY_HEAD_SCALE = 1.0
# final value linear's initialization
VALUE_HEAD_SCALE = 0.1

# Collection and rollout
# complete outer tasks collected per stream before training
EPISODES_PER_STREAM = 1
MAX_CONTEXT = REPEAT_COUNT * (
    MAX_STEPS_PER_EPISODE + 1
)  # every inner step and reset in one outer task
# env steps collected before the first optimizer update
LEARNING_STARTS = 64

# Training
# packed tokens; whole episodes are added until the next one would not fit
TOKEN_BUDGET = 16384
# outer rollout/train cycles
NUM_CYCLES = 2
# optimizer updates per PPO epoch (passed to run_train)
TRAIN_STEPS = 20
# run_train calls per cycle
PPO_EPOCHS = 2

# Optimizer learning rates
# token embeddings and numeric Fourier features
LR_EMBEDDING = 1e-05
# backbone
LR_BACKBONE = 1e-05
# prediction heads and their input RMSNorms
LR_HEADS = 1e-05

# Runtime
ensure_registered()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Build Environment

Online PPO uses the same FrozenLake `RepeatTaskEnv` inside mouse-gym as the DQN
notebooks. The rollout records coordinator, agent, and environment events in that order,
omitting the agent on resets. Each stream’s standalone `RepeatTaskTracker` consumes
outputs, tracks counters and completed statistics, and determines when to act or reset;
the log only records events. The tokenizer reads the source events directly, grouping
tokens by `step_index`.


In [ ]:
def make_lake(*, stream: int):
    def env_fn():
        lake = gym.make(
            "Procedural-FrozenLake-v1",
            width=8,
            height=8,
            max_episode_steps=MAX_STEPS_PER_EPISODE,
            map_seed=stream,
            slippery_success_rate=1.0,
            permute_obs=True,
            permute_actions=True,
            render_mode=None,
            emit_q_star=False,
        )
        env = RepeatTaskEnv(
            lake,
            max_task_episodes=REPEAT_COUNT,
            episode_seed_mode="per_task",
            episode_reset_options={"regenerate_map": False},
        )
        return env

    return env_fn


def make_rollout_env(*, seed_offset: int):
    return make_group_env(
        [
            EnvConfig(
                name=f"proc_frozenlake_ppo_{i}",
                seed=seed_offset + i,
                seed_mode="stream",
                env_fn=make_lake(stream=i),
                reset_options={"regenerate_map": True},
            )
            for i in range(NUM_ENVS)
        ]
    )


# Each on-policy collection starts fresh streams, matching the replay prefix.
env = None

logs = []
trackers = []

## Data Pipeline

Each collection phase creates fresh environment streams and event logs and clears the
on-policy stores. Seeds advance between phases. Every task retains its full policy
context, including inner resets, so replay uses the same history as collection.
`MAX_CONTEXT` must accommodate the worst-case complete task; history is never truncated.


In [ ]:
def action_name(action):
    return ("a", "b", "c", "d")[action]


stores = [Datastore(name=f"proc_frozenlake_ppo_{i}") for i in range(NUM_ENVS)]
contexts = [[] for _ in range(NUM_ENVS)]

# Online decode / on-policy buffer: tokenizer
episode_prompt = (
    "Your job is to predict the future sum of rewards in FrozenLake. "
    "Navigate a grid; reach the goal for reward; a hole ends the "
    "episode with none. Each episode lasts at most "
    f"{MAX_STEPS_PER_EPISODE} steps. You have {REPEAT_COUNT} "
    "episodes to solve the task. The grid is permuted, so squares are "
    "not in order; action ids may be remapped, then rendered as a, b, c, d.\n"
    "Strategy: explore; keep a mental map of what has and has not been "
    "explored; avoid holes you have already fallen in; once you have a "
    "path to the goal, repeat it.\n"
    "At each action-request newline, predict the value of all actions. Format:\n"
    "action letter,observation[,r=reward][,terminated][,truncated].\nx marks no action at the beginning of an episode.\n"
)

tokenizer = Tokenizer(
    tokenizers={"text": PRETRAINED},
    step_field="step_index",
    input_fields=[
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "coordinator",
            "input_field": "_episode_start",
            "format": episode_prompt,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": 'episode {repeat_episode_index+1}\n',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "repeat_episode_index",
            "format": "x",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "agent",
            "input_field": "action",
            "format": '{action_name(action)}',
            "format_functions": {"action_name": action_name},
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "observation",
            "format": ',{observation}',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "reward",
            "format": ',r={reward:g}',
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_terminated",
            "format": ",terminated",
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "inner_truncated",
            "format": ",truncated",
        },
        {
            "type": "token",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": "action_requested",
            "format": "\n",
            "head_output": True,
        },
        {
            "type": "text",
            "tokenizer": "text",
            "provider": "environment",
            "input_field": ["inner_terminated", "inner_truncated"],
            "format": "\n\n",
        },
    ],
    objective_fields=[
        {
            "input_field": "action",
            "missing_value": None,
        },
        {
            "input_field": "reward",
            "missing_value": 0.0,
        },
        {
            "input_field": "_terminated",
            "output_field": "terminated",
            "missing_value": False,
        },
        {
            "input_field": "_truncated",
            "output_field": "truncated",
            "missing_value": False,
        },
        {
            "input_field": "_old_log_prob",
            "output_field": "old_log_prob",
            "missing_value": 0.0,
        },
    ],
)


transform = tokenizer.encode

## Build Encoder and Heads

Call the encoder once, then apply the head(s) to its encodings. Cached inference uses
the same encoder and returns cache and readout-mask metadata. A `ModuleDict` called
`modules` holds references to the online components for checkpointing and mode changes;
it is not called for predictions.


In [ ]:
class Encoder(Model):
    def __init__(
        self, *, pretrained, tokenizer_sources, dtype
    ):
        super().__init__()
        self.embedding = MultiEmbedding(
            embeddings={
                name: Embedding(pretrained=source)
                for name, source in tokenizer_sources.items()
            }
        )
        self.backbone = Backbone(
            pretrained=pretrained,
            dtype=dtype,
        )

    def forward(self, tokens, *, cache=None, use_cache=False):
        if cache is not None and not use_cache:
            raise ValueError("Passing cache= requires use_cache=True.")
        tensors = tokens.to(next(self.parameters()).device)
        token_ids = tensors["token_ids"]
        numeric_values = tensors["numeric_values"]
        numeric_mask = tensors["numeric_mask"]
        sequence_ids = tensors["sequence_ids"]
        head_token_ids = tensors["head_token_mask"].nonzero(as_tuple=True)[0]
        embed = self.embedding(
            token_ids,
            numeric_values=numeric_values,
            numeric_mask=numeric_mask,
            numeric_field_ids=tensors["numeric_field_ids"],
            numeric_field_names=tokens["numeric_field_names"],
            tokenizer_ids=tensors["tokenizer_ids"],
            tokenizer_names=tokens["tokenizer_names"],
        )
        if use_cache:
            embed, lengths, head_token_ids, _ = prepare_decode(
                embeds=embed,
                tokens=tokens,
                sequence_ids=sequence_ids,
                head_token_mask=tensors["head_token_mask"],
            )
            if cache is None:
                cache = DecodeCache(
                    sessions=(
                        self.backbone.decode_session(
                            batch_size=tokens["num_sequences"]
                        ),
                    )
                )
            if len(cache.sessions) != 1:
                raise ValueError("expected one backbone cache session")
            features = self.backbone(embed, cache=cache.sessions[0], lengths=lengths)
            head_gather_ids = head_token_ids.unsqueeze(-1).expand(
                *head_token_ids.shape,
                features.shape[-1],
            )
            h = features.gather(1, head_gather_ids)
        else:
            features = self.backbone(embed, sequence_ids=sequence_ids)
            h = features[head_token_ids]
        return {
            "z": h,
            "cache": cache,
        }


class PolicyHead(nn.Module):
    def __init__(self, *, hidden_dim, num_actions, num_layers, scale):
        super().__init__()
        self.mlp = MLP(
            in_features=hidden_dim,
            out_features=num_actions,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            scale=scale,
            propagate_gradient=1.0,
        )

    def forward(self, z):
        return self.mlp(z)


class ValueHead(nn.Module):
    def __init__(self, *, hidden_dim, num_layers, scale):
        super().__init__()
        self.mlp = MLP(
            in_features=hidden_dim,
            out_features=1,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            scale=scale,
            propagate_gradient=1.0,
        )

    def forward(self, z):
        return self.mlp(z)


encoder = (
    Encoder(
        tokenizer_sources=tokenizer.tokenizer_sources,
        pretrained=PRETRAINED,
        dtype=torch.float32,
    )
    .train()
    .to(device)
)
policy_head = (
    PolicyHead(
        hidden_dim=encoder.backbone.hidden_dim,
        num_actions=MAX_ACTIONS,
        num_layers=HEAD_LAYERS,
        scale=POLICY_HEAD_SCALE,
    )
    .train()
    .to(device)
)
value_head = (
    ValueHead(
        hidden_dim=encoder.backbone.hidden_dim,
        num_layers=HEAD_LAYERS,
        scale=VALUE_HEAD_SCALE,
    )
    .train()
    .to(device)
)

# A checkpoint container; forward calls use the encoder and heads separately.
modules = nn.ModuleDict(
    {"encoder": encoder, "policy_head": policy_head, "value_head": value_head}
)

print(encoder)
print(policy_head)
print(value_head)

## Collect Complete Tasks

Collect one stream at a time, finishing each outer task before moving to the next. The
policy samples only when the environment state requests an action. Each agent event
records the action and its behavior log-probability; reset observations have no agent
event.

A separate KV cache serves each task, grows with its full context, and closes at the
real terminal or truncated observation. No stream is started and left unfinished.


In [ ]:
@torch.no_grad()
def run_rollout(*, env_steps: int, num_episodes: int, max_cache: int) -> int:
    """Collect complete outer tasks, one stream at a time, before training."""
    if num_episodes < 1:
        raise ValueError("num_episodes must be positive")
    if max_cache < REPEAT_COUNT * (MAX_STEPS_PER_EPISODE + 1):
        raise ValueError(
            "max_cache must fit a complete outer task, including reset observations"
        )
    global env, logs, trackers
    if env is not None:
        env.close()
    env = make_rollout_env(seed_offset=env_steps)
    logs = [EventLog(name=name) for name in env.names]
    trackers = [RepeatTaskTracker() for _ in logs]
    for store in stores:
        store.clear()
    modules.eval()
    for i, stream in enumerate(env.envs):
        log = logs[i]
        tracker = trackers[i]
        for _episode in range(num_episodes):
            contexts[i].clear()
            cache = None
            cached_rows = 0
            try:
                while True:
                    event_start = len(log)
                    episode_start = tracker.output is None or tracker.ended
                    episode_index = tracker.next_episode_index
                    if episode_start:
                        coordinator_event = log.event(
                            provider="coordinator",
                            episode_index=episode_index,
                            step_index=(0 if episode_start else tracker.step_index + 1),
                            payload={
                                "_episode_start": True,
                            },
                        )
                        stores[i].append(data=coordinator_event)
                    action = None
                    if tracker.action_requested:
                        pending = [contexts[i][cached_rows:]]
                        inputs, inputs_objective_data = tokenizer.pack_rows(
                            rows=pending
                        )
                        out = encoder(inputs, cache=cache, use_cache=True)
                        cache = out["cache"]
                        cached_rows = len(contexts[i])
                        logits = action_readouts(policy_head(out["z"]), inputs)
                        actions, log_probs = sample_discrete_action(
                            logits=logits, num_actions=MAX_ACTIONS
                        )
                        action = int(actions[0].item())
                        agent_event = log.event(
                            provider="agent",
                            episode_index=episode_index,
                            step_index=tracker.step_index + 1,
                            payload={
                                "action": action,
                                "_old_log_prob": float(log_probs[0].item()),
                            },
                        )
                        stores[i].append(data=agent_event)
                    output = stream.step(action)
                    tracker.update(output=output)

                    environment_event = log.event(
                        provider="environment",
                        episode_index=tracker.episode_index,
                        step_index=tracker.step_index,
                        payload={
                            "observation": tracker.observation,
                            **(
                                {"repeat_episode_index": tracker.repeat_episode_index}
                                if tracker.repeat_episode_start
                                else {}
                            ),
                            **(
                                {"inner_terminated": True}
                                if tracker.inner_terminated
                                else {}
                            ),
                            **(
                                {"inner_truncated": True}
                                if tracker.inner_truncated
                                else {}
                            ),
                            **(
                                {"_q_star": tracker.info["q_star"]}
                                if tracker.action_requested and "q_star" in tracker.info
                                else {}
                            ),
                            **({"_terminated": True} if tracker.terminated else {}),
                            **({"_truncated": True} if tracker.truncated else {}),
                            **(
                                {"reward": tracker.reward}
                                if tracker.reward != 0
                                else {}
                            ),
                            **(
                                {"action_requested": True}
                                if tracker.action_requested
                                else {}
                            ),
                        },
                    )
                    stores[i].append(data=environment_event)
                    contexts[i].extend(log[event_start:])
                    env_steps += 1
                    if tracker.step_index + 1 > max_cache:
                        raise ValueError(
                            "Environment exceeded the configured complete-task context bound"
                        )
                    if tracker.ended:
                        break
            finally:
                if cache is not None:
                    cache.close()
    return env_steps

## Train

After collection, `loader.refresh()` rescans the on-policy stores.
`sample_ranges=sample_ranges` keeps complete outer tasks together. Agent
probabilities enter `objective_data` through `_old_log_prob`; every task ends at a
recorded termination or truncation with zero discount. The objective raises if a sampled
target would require a missing continuation.

The loader returns `inputs` (tokens, embedding features, sequence IDs, and
`head_token_mask`) and `objective_data` (training labels and step mappings).
`objective_data["head_step_ids"][p]` tells the loss which step owns prediction row `p`,
including steps with zero or multiple predictions. Pass this existing mapping to the
objective; no separate prediction-count column is needed. An absent action is
represented by `NaN` in the packed labels.


In [ ]:
def sample_ranges(dataset):
    """Keep all events belonging to one outer episode together."""
    start, previous = 0, None
    for index, episode in enumerate(dataset["episode_index"][:]):
        if episode is not None and episode != previous:
            if index:
                yield start, index
            start, previous = index, episode
    if len(dataset):
        yield start, len(dataset)


def sample_eligible(events):
    """Train only on episodes with a recorded final environment result."""
    last = events[-1]
    return last["provider"] == "environment" and (
        last["payload"].get("_terminated") is True
        or last["payload"].get("_truncated") is True
    )


loader = DataLoader(
    stores=stores,
    token_budget=TOKEN_BUDGET,
    sample_ranges=sample_ranges,
    sample_eligible=sample_eligible,
    weight_mode="per_event",
    num_workers=4,
    transform=tokenizer.encode,
)
objective = PpoObjective(
    reward=None,
    value=None,
    discount=boundary_discount(
        gamma_step=1.0,
        gamma_terminated=0.0,
        gamma_truncated=0.0,
    ),
    gae_lambda=0.95,
    clip_eps=0.2,
    vf_coef=0.5,
    ent_coef=0.01,
    num_actions=MAX_ACTIONS,
)
optimizer = AdamW(
    params=[
        *encoder.parameter_groups(
            lr={"embedding": LR_EMBEDDING, "backbone": LR_BACKBONE}
        ),
        {"params": policy_head.parameters(), "lr": LR_HEADS},
        {"params": value_head.parameters(), "lr": LR_HEADS},
    ],
    weight_decay=0.0,
    betas=(0.9, 0.95),
    eps=1e-08,
)


def run_train(
    *, num_steps: int
) -> tuple[torch.Tensor, dict[str, float | torch.Tensor]]:
    """Refresh the on-policy buffer and run ``num_steps`` PPO optimizer steps."""
    modules.train()
    loader.refresh()
    loss: torch.Tensor | None = None
    metrics: dict[str, float | torch.Tensor] = {}
    for _ in range(num_steps):
        inputs, objective_data = loader.next_batch()
        inputs = inputs.to(device)
        objective_data = objective_data.to(device)
        group_id = objective_data["step_sequence_ids"]
        out = encoder(inputs)
        losses, metrics = objective(
            objective_data=objective_data.to(device),
            group_id=group_id.to(device),
            head_step_ids=objective_data["head_step_ids"],
            predictions=policy_head(out["z"]),
            value_predictions=value_head(out["z"]),
        )
        loss = torch.stack(tuple(losses.values())).sum()
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    assert loss is not None
    return (loss, metrics)

## Run

Each cycle collects `EPISODES_PER_STREAM` complete tasks per stream, trains for
`PPO_EPOCHS` × `TRAIN_STEPS` updates once `LEARNING_STARTS` environment outputs have
been collected, and clears the on-policy stores. Reconstruct this policy head when
evaluating the checkpoint.


In [ ]:
env_steps = 0
grad_steps = 0
for cycle in range(NUM_CYCLES):
    env_steps = run_rollout(
        env_steps=env_steps, num_episodes=EPISODES_PER_STREAM, max_cache=MAX_CONTEXT
    )
    print(f"cycle={cycle} rollout  env_step={env_steps}")
    if env_steps >= LEARNING_STARTS:
        for _epoch in range(PPO_EPOCHS):
            loss, metrics = run_train(num_steps=TRAIN_STEPS)
            grad_steps += TRAIN_STEPS
            print(
                f"cycle={cycle} epoch={_epoch} train    loss={loss.item():.4f}  policy={metrics['policy_loss']:.4f}  value={metrics['value_loss']:.4f}  grad_step={grad_steps}",
            )
        for store in stores:
            store.clear()
        if device.type == "cuda":
            torch.cuda.empty_cache()

loader.close()
if env is not None:
    env.close()
print(f"Online PPO finished ({grad_steps} optimizer steps, {env_steps} env steps).")

## Save To The Hub

The checkpoint contains the encoder and separate head(s) in `modules`. The model weights
and tokenizer packing specification share `MODEL_ID`. Reconstruct the same component
structure before loading. Checkpoints from the old architecture with a projection are
not compatible with this example.


In [ ]:
modules.eval().to("cpu")
model_url = push_model_to_hub(
    model=modules,
    tokenizer=tokenizer,
    repo_id=MODEL_ID,
    private=False,
    clear=True,
)
print(f"Pushed encoder, heads, and tokenizer to {model_url}")